# DiffusionGemma — `google/diffusiongemma-26B-A4B-it`

Works in:
- **GitHub Codespace (Edupack)** — 16-core / 64 GB RAM, CPU inference. In VS Code, click **Select Kernel** and choose **Python 3** before running cells.
- **Google Colab Pro** — L4 (24 GB) or A100 (40 GB) GPU
- **Kaggle** — 2×T4 (30 GB) with `device_map="auto"`

> Free Colab T4 (15 GB) is below the 18 GB minimum — use one of the above instead.

In [ ]:
import shutil, subprocess, psutil

if shutil.which("nvidia-smi"):
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
    print("GPU:", gpu)
else:
    gpu = ""
    print("GPU: None — running in CPU mode")

ram_gb = psutil.virtual_memory().total / 1e9
print(f"RAM: {ram_gb:.0f} GB")
if not gpu and ram_gb < 54:
    print("WARNING: CPU mode needs ~54 GB free RAM. Current RAM may be insufficient.")

In [ ]:
%pip install -q -U "transformers>=4.53.0" accelerate torch ipywidgets psutil

In [ ]:
import torch, transformers
from transformers import DiffusionGemmaForBlockDiffusion, AutoProcessor

MODEL_ID = "google/diffusiongemma-26B-A4B-it"

has_cuda = torch.cuda.is_available()
load_dtype = "auto" if has_cuda else torch.bfloat16

print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)
print("CUDA:", has_cuda)
print(f"Loading {MODEL_ID} with dtype={load_dtype} ...")
model = DiffusionGemmaForBlockDiffusion.from_pretrained(
    MODEL_ID,
    dtype=load_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print("Ready.")
print("Device map:", getattr(model, "hf_device_map", "cpu"))
print("Model dtype:", next(model.parameters()).dtype)

## Basic generation

`num_diffusion_steps` controls the quality/speed tradeoff. On CPU, start small:
- **4–8 steps** — fastest CPU smoke test
- **16 steps** — balanced GPU/default-quality run
- **24 steps** — best quality but slowest

In [ ]:
def generate(prompt: str, max_new_tokens: int = 64, num_diffusion_steps: int = 8) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_diffusion_steps=num_diffusion_steps,
        )
    return processor.decode(output[0], skip_special_tokens=True)


print(generate("Explain diffusion language models in 3 sentences."))

## Multi-turn chat

In [ ]:
history = []

def chat(user_msg: str, max_new_tokens: int = 64, num_diffusion_steps: int = 8) -> str:
    history.append({"role": "user", "content": user_msg})
    inputs = processor.apply_chat_template(
        history,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_diffusion_steps=num_diffusion_steps,
        )
    reply = processor.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    history.append({"role": "assistant", "content": reply})
    return reply


print(chat("What makes diffusion LLMs different from autoregressive ones?"))
print("---")
print(chat("Give a concrete example of where that difference matters."))

## Interactive playground

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

prompt_box = widgets.Textarea(
    value="Write a haiku about masked language modeling.",
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="80px"),
)
max_tokens_slider = widgets.IntSlider(value=64, min=32, max=256, step=32, description="Max tokens")
steps_slider = widgets.IntSlider(value=8, min=4, max=24, step=4, description="Diff. steps")
run_button = widgets.Button(description="Generate", button_style="primary")
out = widgets.Output()

def on_click(_):
    with out:
        clear_output()
        print("Generating...")
        result = generate(prompt_box.value, max_tokens_slider.value, steps_slider.value)
        clear_output()
        print(result)

run_button.on_click(on_click)
display(prompt_box, max_tokens_slider, steps_slider, run_button, out)

## Compare diffusion step counts

See how quality changes from 4 → 16 steps on the same prompt. This is slow on CPU.

In [ ]:
prompt = "Give one surprising fact about language models."

for steps in [4, 8, 16]:
    result = generate(prompt, max_new_tokens=48, num_diffusion_steps=steps)
    print(f"[{steps} steps] {result}")
    print()